In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 accelerate peft wandb datasets
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime

# Fetch secrets from Kaggle securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

# Log in to HuggingFace
login(hf_token, add_to_git_credential=True)

# Log in to Weights & Biases
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
PROJECT_NAME = "price"
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
# Constants
BASE_MODEL = "google/gemma-3-4b-it" 
HF_USER = "sairuthwik112" # CHANGE THIS TO YOUR HF NAME!

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" #

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite" #[cite: 1]
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}" #[cite: 1]
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}" #[cite: 1]

# Hyper-parameters - overall[cite: 1]
EPOCHS = 1 
BATCH_SIZE = 32 
MAX_SEQUENCE_LENGTH = 128 
GRADIENT_ACCUMULATION_STEPS = 1 

# Hyper-parameters - QLoRA[cite: 1]
LORA_R = 32 
LORA_ALPHA = LORA_R * 2 
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"] 
LORA_DROPOUT = 0.1 

# Hyper-parameters - training[cite: 1]
LEARNING_RATE = 1e-4 
WARMUP_RATIO = 0.01 
LR_SCHEDULER_TYPE = 'cosine' 
WEIGHT_DECAY = 0.001 
OPTIMIZER = "paged_adamw_32bit" 

capability = torch.cuda.get_device_capability() #[cite: 1]
use_bf16 = capability[0] >= 8 #[cite: 1]

# Tracking[cite: 1]
VAL_SIZE = 500 
LOG_STEPS = 5 
SAVE_STEPS = 100

In [ ]:
dataset = load_dataset(DATASET_NAME) #[cite: 1]
train = dataset['train'] #[cite: 1]
val = dataset['val'].select(range(VAL_SIZE)) #[cite: 1]

wandb.init(project=PROJECT_NAME, name=RUN_NAME) #[cite: 1]

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
) #[cite: 1]

# Load the Tokenizer and the Model[cite: 1]
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
) #[cite: 1]
base_model.generation_config.pad_token_id = tokenizer.pad_token_id #[cite: 1]

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB") #[cite: 1]

In [ ]:
from transformers import DataCollatorForLanguageModeling
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# 1. Define the Custom Collator for Gemma 3
class Gemma3TextCollator:
    def __init__(self, tokenizer):
        self.collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        
    def __call__(self, features):
        # Filter out hidden columns like 'completion_mask' that break padding
        cleaned_features = []
        for feature in features:
            cleaned = {k: v for k, v in feature.items() if k in ["input_ids", "attention_mask", "labels"]}
            cleaned_features.append(cleaned)
            
        # Pad the batch
        batch = self.collator(cleaned_features)
        
        # Inject all-zero token_type_ids for text-only inputs if missing
        if "token_type_ids" not in batch:
            batch["token_type_ids"] = batch["input_ids"].new_zeros(batch["input_ids"].shape)
        return batch

# 2. LoRA Parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# 3. Training parameters
BATCH_SIZE = 2                            
GRADIENT_ACCUMULATION_STEPS = 16          

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,          
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="prompt",
    remove_unused_columns=True
)

# 4. Trainer Initialization
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
    processing_class=tokenizer, 
    data_collator=Gemma3TextCollator(tokenizer) 
)

In [ ]:
# Fine-tune![cite: 1]
fine_tuning.train()

# Push our fine-tuned model to Hugging Face[cite: 1]
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

# Close the Weights & Biases run properly
if LOG_TO_WANDB:
    wandb.finish()